In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# -*- coding: utf-8 -*-
import numpy as np
import pandas as pd
import cv2
import re
from pathlib import Path
from collections import defaultdict
import albumentations as A
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import random

# Set all seeds for reproducibility
random.seed(42)
np.random.seed(42)

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Configuration
data_dir = Path('/content/drive/MyDrive/Alzheimers /Data')
IMG_SIZE = 128

class_names = ['Non Demented', 'Very mild Dementia', 'Mild Dementia', 'Moderate Dementia']

label_map = {
    'Non Demented': 0,
    'Very mild Dementia': 1,
    'Mild Dementia': 2,
    'Moderate Dementia': 3
}

train_targets = {
    'Non Demented': 3000,
    'Very mild Dementia': 3000,
    'Mild Dementia': 3000,
    'Moderate Dementia': 1464
}

# Augmentation pipeline
augmentation = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=10, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=10, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3)
])

def extract_subject_id(filename):
    match = re.search(r'OAS1_\d{4}', filename)
    return match.group(0) if match else None

def load_and_preprocess_image(image_path):
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.float32) / 255.0
    return img

# Step 1: Group images by subject
print("Step 1: Grouping images by subject")
subject_groups = defaultdict(lambda: defaultdict(list))

for class_name in class_names:
    class_path = data_dir / class_name
    image_files = list(class_path.glob('*.jpg'))

    for img_path in image_files:
        subject_id = extract_subject_id(img_path.name)
        if subject_id:
            subject_groups[class_name][subject_id].append(img_path)

# Step 2: Subject-level 70/15/15 split
print("\nStep 2: Subject-level split (70/15/15)")
splits = {class_name: {'train': [], 'val': [], 'test': []} for class_name in class_names}

for class_name in class_names:
    subjects = list(subject_groups[class_name].keys())
    n_subjects = len(subjects)

    if n_subjects < 3:
        # Edge case: too few subjects to split properly
        train_subjects = subjects
        val_subjects = []
        test_subjects = []
    else:
        train_subjects, temp_subjects = train_test_split(subjects, train_size=0.7, random_state=42)

        if len(temp_subjects) < 2:
            # Not enough subjects for val/test split
            val_subjects = temp_subjects
            test_subjects = []
        else:
            val_subjects, test_subjects = train_test_split(temp_subjects, train_size=0.5, random_state=42)

    for subj in train_subjects:
        splits[class_name]['train'].extend(subject_groups[class_name][subj])
    for subj in val_subjects:
        splits[class_name]['val'].extend(subject_groups[class_name][subj])
    for subj in test_subjects:
        splits[class_name]['test'].extend(subject_groups[class_name][subj])

    print(f"{class_name}: Train={len(splits[class_name]['train'])}, Val={len(splits[class_name]['val'])}, Test={len(splits[class_name]['test'])}")

# Step 3: Balance training set only
print("\nStep 3: Balancing training set")

def balance_class_training(class_name, target_count):
    train_images = splits[class_name]['train']
    current_count = len(train_images)
    label = label_map[class_name]

    print(f"\n{class_name}: {current_count} -> {target_count}")

    if current_count == target_count:
        X_class = [load_and_preprocess_image(img_path) for img_path in tqdm(train_images, desc=f"Loading {class_name}")]
        y_class = [label] * target_count
        return X_class, y_class

    elif current_count > target_count:
        # Undersample using subject-level greedy selection
        subjects_in_train = defaultdict(list)
        for img_path in train_images:
            subj = extract_subject_id(img_path.name)
            subjects_in_train[subj].append(img_path)

        subject_list = list(subjects_in_train.keys())
        random.shuffle(subject_list)

        selected_images = []
        for subj in subject_list:
            selected_images.extend(subjects_in_train[subj])
            if len(selected_images) >= target_count:
                break

        # Trim to exact target
        if len(selected_images) > target_count:
            random.shuffle(selected_images)
            selected_images = selected_images[:target_count]

        X_class = [load_and_preprocess_image(img_path) for img_path in tqdm(selected_images, desc=f"Loading {class_name}")]
        y_class = [label] * len(X_class)
        return X_class, y_class

    else:
        # Augment to target
        X_original = [load_and_preprocess_image(img_path) for img_path in tqdm(train_images, desc=f"Loading {class_name}")]
        y_class = [label] * current_count

        needed = target_count - current_count
        print(f"  Augmenting {needed} images")

        for _ in tqdm(range(needed), desc=f"Augmenting {class_name}"):
            idx = np.random.randint(0, current_count)
            img = X_original[idx]

            img_uint8 = (img * 255).astype(np.uint8)
            augmented = augmentation(image=img_uint8)
            aug_img = augmented['image'].astype(np.float32) / 255.0

            X_original.append(aug_img)
            y_class.append(label)

        return X_original, y_class

# Balance each class in training set
X_train_all = []
y_train_all = []

for class_name in class_names:
    X_class, y_class = balance_class_training(class_name, train_targets[class_name])
    X_train_all.extend(X_class)
    y_train_all.extend(y_class)

X_train = np.array(X_train_all, dtype=np.float32)
y_train = np.array(y_train_all, dtype=np.int32)

# Shuffle training set
print("\nShuffling training set")
shuffle_idx = np.random.permutation(len(X_train))
X_train = X_train[shuffle_idx]
y_train = y_train[shuffle_idx]

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

# Step 4: Load validation set
print("\nStep 4: Loading validation set")
X_val_all = []
y_val_all = []

for class_name in class_names:
    val_images = splits[class_name]['val']
    label = label_map[class_name]

    for img_path in tqdm(val_images, desc=f"Loading val {class_name}"):
        img = load_and_preprocess_image(img_path)
        X_val_all.append(img)
        y_val_all.append(label)

X_val = np.array(X_val_all, dtype=np.float32)
y_val = np.array(y_val_all, dtype=np.int32)

# Shuffle validation set
shuffle_idx = np.random.permutation(len(X_val))
X_val = X_val[shuffle_idx]
y_val = y_val[shuffle_idx]

print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")

# Step 5: Load test set
print("\nStep 5: Loading test set")
X_test_all = []
y_test_all = []

for class_name in class_names:
    test_images = splits[class_name]['test']
    label = label_map[class_name]

    for img_path in tqdm(test_images, desc=f"Loading test {class_name}"):
        img = load_and_preprocess_image(img_path)
        X_test_all.append(img)
        y_test_all.append(label)

X_test = np.array(X_test_all, dtype=np.float32)
y_test = np.array(y_test_all, dtype=np.int32)

# Shuffle test set
shuffle_idx = np.random.permutation(len(X_test))
X_test = X_test[shuffle_idx]
y_test = y_test[shuffle_idx]

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

# Step 6: Save all arrays
print("\nStep 6: Saving arrays")
np.save("X_train_balanced.npy", X_train)
np.save("y_train_balanced.npy", y_train)
np.save("X_val.npy", X_val)
np.save("y_val.npy", y_val)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

print("\nDone! Files saved:")
print(f"  X_train_balanced.npy: {X_train.shape}")
print(f"  y_train_balanced.npy: {y_train.shape}")
print(f"  X_val.npy: {X_val.shape}")
print(f"  y_val.npy: {y_val.shape}")
print(f"  X_test.npy: {X_test.shape}")
print(f"  y_test.npy: {y_test.shape}")

print("\nTraining set class distribution:")
for i in range(4):
    count = np.sum(y_train == i)
    print(f"  Class {i}: {count} images")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Step 1: Grouping images by subject


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)



Step 2: Subject-level split (70/15/15)
Non Demented: Train=4868, Val=810, Test=1297
Very mild Dementia: Train=2345, Val=493, Test=518
Mild Dementia: Train=3355, Val=732, Test=915
Moderate Dementia: Train=488, Val=0, Test=0

Step 3: Balancing training set

Non Demented: 4868 -> 3000


Loading Non Demented:   0%|          | 0/3000 [00:00<?, ?it/s]


Very mild Dementia: 2345 -> 3000


...

Loading test Non Demented:   0%|          | 0/1297 [00:00<?, ?it/s]

Loading test Very mild Dementia:   0%|          | 0/518 [00:00<?, ?it/s]

Loading test Mild Dementia:   0%|          | 0/915 [00:00<?, ?it/s]

Loading test Moderate Dementia: 0it [00:00, ?it/s]

X_test shape: (2730, 128, 128, 3)
y_test shape: (2730,)

Step 6: Saving arrays

Done! Files saved:
  X_train_balanced.npy: (10464, 128, 128, 3)
  y_train_balanced.npy: (10464,)
  X_val.npy: (2035, 128, 128, 3)
  y_val.npy: (2035,)
  X_test.npy: (2730, 128, 128, 3)
  y_test.npy: (2730,)

Training set class distribution:
  Class 0: 3000 images
  Class 1: 3000 images
  Class 2: 3000 images
  Class 3: 1464 images


In [ ]:
from google.colab import drive
import shutil
import os

# Define your target folder path
target_folder = '/content/drive/MyDrive/Alzheimers /Recent Data'  # Change this path
os.makedirs(target_folder, exist_ok=True)

# List of your .npy files
npy_files = [
    "X_train_balanced.npy",
    "y_train_balanced.npy",
    "X_val.npy",
    "y_val.npy",
    "X_test.npy",
    "y_test.npy"
]


# Copy files to Drive
for file in npy_files:
    shutil.copy(file, target_folder)
    print(f'Saved {file} to {target_folder}')

Saved X_train_balanced.npy to /content/drive/MyDrive/Alzheimers /Recent Data
Saved y_train_balanced.npy to /content/drive/MyDrive/Alzheimers /Recent Data
Saved X_val.npy to /content/drive/MyDrive/Alzheimers /Recent Data
Saved y_val.npy to /content/drive/MyDrive/Alzheimers /Recent Data
Saved X_test.npy to /content/drive/MyDrive/Alzheimers /Recent Data
Saved y_test.npy to /content/drive/MyDrive/Alzheimers /Recent Data
